# 00 - Setup: Live Or Offline

This notebook checks the AcmeCloud fixture and initializes the shared Metatate client.

Offline mode is the default. It replays RECORDED Metatate Cloud answers (captured
from a live workspace by `scripts/record_offline_fixtures.py`), so what you study
offline is exactly what the live endpoint returns — typed answers with
`state`, lowercase decision vocabulary, structured conditions, and publication provenance.

Live mode calls your Metatate Cloud workspace's MCP endpoint (no account yet? create one free at [app.getmetatate.com/sign-up?ref=examples](https://app.getmetatate.com/sign-up?ref=examples) and load the AcmeCloud demo from the dashboard's **"New here?" banner → Load the demo**). Set `METATATE_EXAMPLES_MODE=live`, export `METATATE_MCP_URL` and your access token, then start Jupyter — see [docs/live-mode-saas.md](../docs/live-mode-saas.md).


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None, schema="public"):
    ref = {"database": "acmecloud_demo", "schema": schema, "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


## Load Synthetic Tables


In [ ]:
table_dir = repo_root / "sample-data" / "acmecloud" / "tables"
tables = {}
for path in sorted(table_dir.glob("*.csv")):
    tables[path.stem] = pd.read_csv(path)
    print(f"{path.name}: {len(tables[path.stem])} rows")


In [ ]:
tables["customers"].head()


## Discover Governed Context

`discover_context` lists everything the CURRENT publication governs — each asset
carries its instruction count and the canonical scenario keys it can answer.


In [ ]:
discovery = client.discover_context()
print(f"state: {discovery['state']}")
print(f"publication: {discovery['publication']['publication_id']}")
pd.DataFrame(
    [
        {
            "table": entry["ref"]["table"],
            "column": entry["ref"].get("column"),
            "instructions": entry["instruction_count"],
            "scenarios": ", ".join(entry["scenario_keys"]),
        }
        for entry in discovery["assets"]
    ]
)
